# Marine Domain RAG — 1) End-to-End 파이프라인 데모

본 노트북은 **회사 NDA 로 인해 본인이 별도로 재현한 데모**입니다.
데이터는 모두 공개 국가법령정보 OPEN API 만 사용합니다.

흐름:
1. 국가법령정보 OPEN API 로 해수부 + 해경 소관 법령 수집 (이미 `data/raw/laws/`)
2. 조문 단위 파싱
3. 한국어 sentence-transformers + FAISS + BM25 인덱스
4. GraphRAG 구축 (조문↔용어 그래프)
5. LangGraph 워크플로우 + EXAONE-3.5-2.4B-Instruct (Mac MPS) 답변

In [ ]:
import json, pandas as pd
from pathlib import Path
from marine_domain_rag.config import Config
from marine_domain_rag.parsing.article_parser import parse_laws
from marine_domain_rag.indexing.embed_index import load_index, build_index
from marine_domain_rag.graph.builder import build_graph, save_graph, load_graph
from marine_domain_rag.langgraph_app.workflow import build_app
from marine_domain_rag.llm.exaone_loader import load_llm

cfg = Config.load('../configs/default.yaml')
raw_dir = Path('../') / cfg.get('paths','raw_laws_dir')
laws = [json.load(open(p, encoding='utf-8')) for p in sorted(raw_dir.glob('*.json'))]
print('laws collected:', len(laws))
df = parse_laws(laws)
print('articles parsed:', len(df))
df.head(3)

In [ ]:
# 인덱스 로드 (이미 빌드되어 있다면), 없으면 빌드
idx_dir = Path('../') / cfg.get('paths','faiss_dir')
if (idx_dir / 'meta.json').exists():
    retriever = load_index(idx_dir)
else:
    retriever = build_index(df, cfg.get('embedding','model_name'),
                            int(cfg.get('embedding','batch_size')),
                            int(cfg.get('embedding','max_seq_length')),
                            idx_dir,
                            float(cfg.get('retrieval','bm25_weight')),
                            float(cfg.get('retrieval','embed_weight')))
retriever.search('해양경찰청장의 임용권', k=3)

In [ ]:
# GraphRAG
graph_path = Path('../') / cfg.get('paths','graph_path')
if graph_path.exists():
    g = load_graph(graph_path)
else:
    g = build_graph(df,
                    min_term_freq=int(cfg.get('graph','min_term_frequency')),
                    top_terms_per_article=int(cfg.get('graph','top_terms_per_article')))
print('nodes', g.number_of_nodes(), 'edges', g.number_of_edges())

In [ ]:
# LangGraph + LLM (EXAONE 또는 mock)
llm = load_llm(provider='mock', model_name=cfg.get('llm','model_name'))  # exaone 으로 바꿔도 됨
app = build_app(retriever, g, llm, top_k=int(cfg.get('retrieval','top_k')))
out = app.invoke({'question': '해양경찰청 직원의 복제 기준은 어떻게 정해지나요?'})
print('--- ANSWER ---')
print(out.get('answer') if isinstance(out, dict) else out.answer)
print('--- CITATIONS ---')
for c in (out.get('citations') if isinstance(out, dict) else out.citations):
    print(f"- {c['law_name']} 제{c['article_no']}조 (score={c['score']:.3f})")

In [ ]:
# 평가
from marine_domain_rag.evaluation.qa_eval import evaluate, load_suite
suite = load_suite('../tests/qa_pairs.yaml')
res = evaluate(app, suite, k=5)
print(res)